In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hope-english/english_hope_dev.csv
/kaggle/input/hope-english/english_hope_test.csv
/kaggle/input/hope-english/english_hope_train.csv
/kaggle/input/indic-bert/indic-bert/config.json
/kaggle/input/indic-bert/indic-bert/spiece.vocab
/kaggle/input/indic-bert/indic-bert/spiece.model
/kaggle/input/indic-bert/indic-bert/README.md
/kaggle/input/indic-bert/indic-bert/tf_model.ckpt.meta
/kaggle/input/indic-bert/indic-bert/tf_model.ckpt.data-00000-of-00001
/kaggle/input/indic-bert/indic-bert/pytorch_model.bin
/kaggle/input/indic-bert/indic-bert/.gitattributes
/kaggle/input/indic-bert/indic-bert/tf_model.ckpt.index
/kaggle/input/indic-bert/indic-bert/.git/config
/kaggle/input/indic-bert/indic-bert/.git/packed-refs
/kaggle/input/indic-bert/indic-bert/.git/HEAD
/kaggle/input/indic-bert/indic-bert/.git/index
/kaggle/input/indic-bert/indic-bert/.git/description
/kaggle/input/indic-bert/indic-bert/.git/info/exclude
/kaggle/input/indic-bert/indic-bert/.git/refs/heads/main
/kaggle/input/ind

In [2]:
import pandas as pd

train = pd.read_csv("/kaggle/input/hope-english/english_hope_train.csv", sep="\t", header=None)
dev   = pd.read_csv("/kaggle/input/hope-english/english_hope_dev.csv", sep="\t", header=None)
test  = pd.read_csv("/kaggle/input/hope-english/english_hope_test.csv", sep="\t", header=None)

train.columns = ["raw"]
dev.columns = ["raw"]
test.columns = ["raw"]


In [3]:
def clean_split(df):
    df = df.copy()

    # split by ";" into max 3 parts
    parts = df["raw"].str.split(";", expand=True)

    # parts[0] = text  
    # parts[1] = label
    df["text"] = parts[0]
    df["label"] = parts[1]

    # drop rows without labels
    df = df.dropna(subset=["label"])
    
    # keep only text + label
    df = df[["text", "label"]]
    return df

train = clean_split(train)
dev = clean_split(dev)
test = clean_split(test)

print(train.head())
print(train.shape)


                                                text            label
0  these tiktoks radiate gay chaotic energy and i...  Non_hope_speech
1  @Champions Again He got killed for using false...  Non_hope_speech
2               It's not that all lives don't matter  Non_hope_speech
3  Is it really that difficult to understand? Bla...  Non_hope_speech
4  Whenever we say black isn't that racists?  Why...  Non_hope_speech
(22762, 2)


In [4]:
import re

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+|www\S+", "", t)       # remove URLs
    t = re.sub(r"@\w+", "", t)                # remove usernames
    t = re.sub(r"\s+", " ", t).strip()        # remove extra spaces
    return t

train["text"] = train["text"].apply(clean_text)
dev["text"] = dev["text"].apply(clean_text)
test["text"] = test["text"].apply(clean_text)


In [5]:
label_map = {
    "Hope_speech": 1,
    "Non_hope_speech": 0
}

train["label"] = train["label"].map(label_map)
dev["label"]   = dev["label"].map(label_map)
test["label"]  = test["label"].map(label_map)


In [6]:
train = train.dropna(subset=["label"])
dev   = dev.dropna(subset=["label"])
test  = test.dropna(subset=["label"])


In [7]:
print(train["label"].value_counts())


label
0.0    20700
1.0     1945
Name: count, dtype: int64


In [8]:
# !pip install sentence-transformers

In [9]:
from sentence_transformers import SentenceTransformer

# Load the T5 sentence embedding model
# You can use 'sentence-t5-base' or 'sentence-t5-small' for faster computation
model = SentenceTransformer('sentence-t5-base')

# Suppose your text data is in a pandas DataFrame called train, dev, test
train_texts = train["text"].tolist()
dev_texts   = dev["text"].tolist()
test_texts  = test["text"].tolist()

# Encode the texts into sentence embeddings
# Output: numpy array of shape (num_samples, embedding_dim)
train_embeddings = model.encode(train_texts, batch_size=32, show_progress_bar=True)
dev_embeddings   = model.encode(dev_texts, batch_size=32, show_progress_bar=True)
test_embeddings  = model.encode(test_texts, batch_size=32, show_progress_bar=True)

print("Train embeddings shape:", train_embeddings.shape)
print("Dev embeddings shape:", dev_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

2025-12-25 14:07:26.453624: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766671646.635148      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766671646.686993      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

2_Dense/pytorch_model.bin:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

Batches:   0%|          | 0/708 [00:00<?, ?it/s]

Batches:   0%|          | 0/89 [00:00<?, ?it/s]

Batches:   0%|          | 0/89 [00:00<?, ?it/s]

Train embeddings shape: (22645, 768)
Dev embeddings shape: (2830, 768)
Test embeddings shape: (2828, 768)


In [10]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert embeddings to PyTorch tensors
X_train = torch.tensor(train_embeddings, dtype=torch.float32)
X_dev   = torch.tensor(dev_embeddings, dtype=torch.float32)
X_test  = torch.tensor(test_embeddings, dtype=torch.float32)

# Convert labels to tensors
y_train = torch.tensor(train["label"].values, dtype=torch.long)
y_dev   = torch.tensor(dev["label"].values, dtype=torch.long)
y_test  = torch.tensor(test["label"].values, dtype=torch.long)

# Create TensorDatasets
train_dataset = TensorDataset(X_train, y_train)
dev_dataset   = TensorDataset(X_dev, y_dev)
test_dataset  = TensorDataset(X_test, y_test)

# Create DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train loader batches:", len(train_loader))
print("Dev loader batches:", len(dev_loader))
print("Test loader batches:", len(test_loader))


Train loader batches: 708
Dev loader batches: 89
Test loader batches: 89


In [11]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

indic_model_path = "/kaggle/input/indic-bert/indic-bert"  # UPDATE THIS PATH

indic_tokenizer = AutoTokenizer.from_pretrained(indic_model_path)
indic_model = AutoModel.from_pretrained(indic_model_path).to(device)

indic_model.eval()
print("IndicBERT loaded")


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

IndicBERT loaded


In [12]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)


In [13]:
def extract_indicbert_embeddings(texts, batch_size=32):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        encoded = indic_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            output = indic_model(**encoded)

        embeddings = mean_pooling(output, encoded["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings).numpy()


In [14]:
train_indic_emb = extract_indicbert_embeddings(train_texts)
dev_indic_emb   = extract_indicbert_embeddings(dev_texts)
test_indic_emb  = extract_indicbert_embeddings(test_texts)

print(train_indic_emb.shape)


(22645, 768)


In [15]:
X_train = np.concatenate([train_embeddings, train_indic_emb], axis=1)
X_dev   = np.concatenate([dev_embeddings, dev_indic_emb], axis=1)
X_test  = np.concatenate([test_embeddings, test_indic_emb], axis=1)

print(X_train.shape)


(22645, 1536)


In [16]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_dev   = torch.tensor(X_dev, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(train["label"].values, dtype=torch.long)
y_dev   = torch.tensor(dev["label"].values, dtype=torch.long)
y_test  = torch.tensor(test["label"].values, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
dev_loader   = DataLoader(TensorDataset(X_dev, y_dev), batch_size=32)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=32)



In [17]:
print(X_train.shape)
print(X_dev.shape)
print(X_test.shape)

torch.Size([22645, 1536])
torch.Size([2830, 1536])
torch.Size([2828, 1536])


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class HopeSpeechCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # FC layer (paper: 1536 → 1536)
        self.fc1 = nn.Linear(1536, 1536)
        self.dropout = nn.Dropout(0.25)

        # CNN blocks (paper specs)
        self.conv1 = nn.Conv1d(1, 64, kernel_size=5)
        self.pool1 = nn.MaxPool1d(4)

        self.conv2 = nn.Conv1d(64, 64, kernel_size=5)
        self.pool2 = nn.MaxPool1d(4)

        self.conv3 = nn.Conv1d(64, 64, kernel_size=5)
        self.pool3 = nn.MaxPool1d(4)

        # 🔥 compute flatten size dynamically
        self.flatten_dim = self._get_flatten_dim()

        # Final classifier
        self.fc_out = nn.Linear(self.flatten_dim, 2)

    def _get_flatten_dim(self):
        with torch.no_grad():
            x = torch.zeros(1, 1536)
            x = F.relu(self.fc1(x))
            x = x.unsqueeze(1)

            x = self.pool1(F.relu(self.conv1(x)))
            x = self.pool2(F.relu(self.conv2(x)))
            x = self.pool3(F.relu(self.conv3(x)))

            return x.view(1, -1).shape[1]

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = x.unsqueeze(1)

        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)
        return self.fc_out(x)


In [19]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, dataloader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return acc, macro_f1


In [20]:
def train_model(model, train_loader, dev_loader, epochs=10, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        dev_acc, dev_f1 = evaluate(model, dev_loader, device)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_loss:.4f}")
        print(f"Dev Accuracy: {dev_acc:.4f}")
        print(f"Dev Macro-F1: {dev_f1:.4f}")

        # Save best model (based on Macro-F1)
        if dev_f1 > best_f1:
            best_f1 = dev_f1
            torch.save(model.state_dict(), "best_model.pt")
            print("✅ Best model saved")

    print(f"\n🏁 Best Dev Macro-F1: {best_f1:.4f}")


In [21]:
model = HopeSpeechCNN()
train_model(model, train_loader, dev_loader, epochs=10, lr=1e-4)



Epoch 1/10
Train Loss: 0.2655
Dev Accuracy: 0.9254
Dev Macro-F1: 0.7395
✅ Best model saved

Epoch 2/10
Train Loss: 0.1821
Dev Accuracy: 0.8452
Dev Macro-F1: 0.7127

Epoch 3/10
Train Loss: 0.1769
Dev Accuracy: 0.8481
Dev Macro-F1: 0.7165

Epoch 4/10
Train Loss: 0.1724
Dev Accuracy: 0.9078
Dev Macro-F1: 0.7788
✅ Best model saved

Epoch 5/10
Train Loss: 0.1695
Dev Accuracy: 0.8770
Dev Macro-F1: 0.7472

Epoch 6/10
Train Loss: 0.1670
Dev Accuracy: 0.8558
Dev Macro-F1: 0.7255

Epoch 7/10
Train Loss: 0.1664
Dev Accuracy: 0.7898
Dev Macro-F1: 0.6631

Epoch 8/10
Train Loss: 0.1632
Dev Accuracy: 0.8788
Dev Macro-F1: 0.7521

Epoch 9/10
Train Loss: 0.1638
Dev Accuracy: 0.8880
Dev Macro-F1: 0.7615

Epoch 10/10
Train Loss: 0.1614
Dev Accuracy: 0.7625
Dev Macro-F1: 0.6407

🏁 Best Dev Macro-F1: 0.7788


In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = HopeSpeechCNN().to(device)
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()


HopeSpeechCNN(
  (fc1): Linear(in_features=1536, out_features=1536, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
  (conv1): Conv1d(1, 64, kernel_size=(5,), stride=(1,))
  (pool1): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,))
  (pool2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(64, 64, kernel_size=(5,), stride=(1,))
  (pool3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  (fc_out): Linear(in_features=1408, out_features=2, bias=True)
)

In [23]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

def evaluate_test(model, dataloader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    weighted_precision = precision_score(all_labels, all_preds, average="weighted")
    weighted_recall    = recall_score(all_labels, all_preds, average="weighted")
    weighted_f1        = f1_score(all_labels, all_preds, average="weighted")

    return acc, macro_f1, weighted_precision, weighted_recall, weighted_f1, all_labels, all_preds


In [24]:
acc, macro_f1, w_prec, w_rec, w_f1, y_true, y_pred = evaluate_test(
    model, test_loader, device
)

print("Test Set Results")
print(f"Accuracy          : {acc:.4f}")
print(f"Macro F1-Score    : {macro_f1:.4f}")
print(f"Weighted Precision: {w_prec:.4f}")
print(f"Weighted Recall   : {w_rec:.4f}")
print(f"Weighted F1-Score : {w_f1:.4f}")


Test Set Results
Accuracy          : 0.9070
Macro F1-Score    : 0.7661
Weighted Precision: 0.9313
Weighted Recall   : 0.9070
Weighted F1-Score : 0.9159


In [25]:
print("\n Classification Report (Test Set):\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Non-Hope", "Hope"]
))



 Classification Report (Test Set):

              precision    recall  f1-score   support

    Non-Hope       0.97      0.92      0.95      2581
        Hope       0.48      0.75      0.58       247

    accuracy                           0.91      2828
   macro avg       0.73      0.84      0.77      2828
weighted avg       0.93      0.91      0.92      2828

